# 🏥 Intelligent Home-Based Post-Surgical Wound Healing Monitoring with Computer Vision

**Author:** Arthur Bezerra Calado (abc4@cin.ufpe.br)  
**Institution:** Universidade Federal de Pernambuco - UFPE  
**Course:** Deep Learning - Draft 3 (Enhanced Analysis)  

## 📋 Execution Instructions

This notebook implements advanced corrections based on advisor feedback. To run:
1. Ensure GPU runtime is enabled (Runtime → Change runtime type → GPU)
2. Install dependencies: `!pip install -q albumentations statsmodels scikit-learn tensorflow`
3. Run cells sequentially (Shift+Enter) from top to bottom
4. New analysis sections are marked with
5. Class 'pressure' rebalanced with weight=2.0 and reduced oversampling

**Note:** This notebook preserves the original implementation and adds targeted corrections. New functions are documented with technical justifications.

## Diário de Experimentos (Resumo)
- **v1 (Baseline):** MobileNetV2 padrão → 62.5% acurácia
- **v2:** + Focal Loss + Oversampling → 66.8% acurácia (+4.3pp)
- **v3:** + Mixup + TTA → 69.02% acurácia (+2.2pp)  
- **v4 (Atual):** EfficientNetB0 + Cosine Annealing → 81.52% acurácia (+12.5pp)
- **v5 (Draft 3):** + Análises Estatísticas + Qualitativas (este notebook)
- **v6 (Correções):** Investigação de fine-tuning, análise de viés de classe, calibração, thresholds otimizados, remoção da phase2 para simplificar (estava atrapalhando)

## 1. Environment Setup and Imports

In [ ]:
# Install required packages
!pip install -q albumentations statsmodels

In [ ]:
import os
import sys
import shutil
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path
from typing import Optional, List, Dict, Tuple
from collections import Counter
import random
import time

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks, backend as K
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0, DenseNet121
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.utils import to_categorical, Sequence
from tensorflow.keras.models import Model, load_model

# Scikit-learn imports
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    precision_recall_curve,
    brier_score_loss
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# Image processing
import cv2
from PIL import Image
import glob

# Statistical tests
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

# Set random seeds
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print("=" * 70)
print("ENVIRONMENT SETUP COMPLETE")
print("=" * 70)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"NumPy version: {np.__version__}")
print("=" * 70)

## 2. Configuration and Hyperparameters

In [ ]:
class Config:
    """
    Configuration class.
    Enhanced with ablation tracking, statistical analysis, and training fixes.
    """

    # Dataset configuration
    IMG_SIZE = 224
    BATCH_SIZE = 16
    NUM_CLASSES = 4
    CLASS_NAMES = ['diabetic', 'pressure', 'surgical', 'venous']

    # Folder to class mapping
    FOLDER_TO_CLASS = {
        'D': 'diabetic',
        'P': 'pressure',
        'S': 'surgical',
        'V': 'venous'
    }
    IGNORE_FOLDERS = ['BG', 'N']

    # Automatic logit adjustment
    APPLY_LOGIT_ADJUSTMENT = False
    LOGIT_ADJUSTMENT_TAU = 1.0

    # Training configuration
    EPOCHS_PHASE1 = 40
    LEARNING_RATE = 5e-4
    LEARNING_RATE_FINETUNE = 1e-7
    WARMUP_EPOCHS = 5
    EARLY_STOPPING_PATIENCE = 20
    REDUCE_LR_PATIENCE = 7
    REDUCE_LR_FACTOR = 0.5
    MIN_LR = 1e-8

    # Fine-tuning stabilization
    UNFREEZE_LR_FACTOR = 0.05  # Fator de redução de LR ao descongelar
    DISCRIMINATIVE_LR_MULTIPLIER = 10.0  # Multiplicador para camadas descongeladas

    # Data split
    VAL_SPLIT = 0.15

    # Augmentation
    ROTATION_RANGE = 15
    WIDTH_SHIFT = 0.1
    HEIGHT_SHIFT = 0.1
    SHEAR_RANGE = 0.1
    ZOOM_RANGE = 0.1
    HORIZONTAL_FLIP = True
    VERTICAL_FLIP = False
    BRIGHTNESS_RANGE = [0.9, 1.1]

    # Model architecture
    DROPOUT_RATE = 0.4
    L2_REGULARIZATION = 0.0001
    DENSE_UNITS_1 = 512
    DENSE_UNITS_2 = 256
    UNFREEZE_LAYERS = 30

    # Advanced features
    USE_FOCAL_LOSS = True
    FOCAL_GAMMA = 3.0
    USE_MIXUP = True
    MIXUP_ALPHA = 0.2
    USE_TTA = True
    TTA_STEPS = 10
    USE_ENSEMBLE = True
    LABEL_SMOOTHING = 0.1
    OVERSAMPLE_MINORITY = True

    # Draft 3 Features
    USE_CUTMIX = False
    CUTMIX_ALPHA = 1.0
    N_MCMC_STEPS = 10

    # Calibration and thresholding
    APPLY_CALIBRATION = True
    CALIBRATION_METHOD = 'temperature_scaling'  # 'platt', 'isotonic'
    OPTIMIZE_THRESHOLDS = True
    THRESHOLD_METRIC = 'f1'  # 'f1', 'precision', 'recall'

    # Visualização
    VISUALIZATION_CONFIDENCE_THRESHOLD = 0.5

    # Paths
    RAW_DATA_DIR = '/content/raw_data'
    DATA_DIR = '/content/wound_dataset'
    MODEL_SAVE_DIR = '/content/models'
    RESULTS_DIR = '/content/results'
    REPO_DIR = '/content/wound-classification-repo'

    @classmethod
    def display(cls):
        print("\n" + "=" * 70)
        print("CONFIGURATION PARAMETERS")
        print("=" * 70)
        for attr in dir(cls):
            if not attr.startswith('_') and not callable(getattr(cls, attr)):
                print(f"  {attr}: {getattr(cls, attr)}")
        print("=" * 70 + "\n")


Config.display()

## 3. Focal Loss Implementation

In [ ]:
class FocalLoss(keras.losses.Loss):
    """Focal Loss for handling class imbalance."""

    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        if self.label_smoothing > 0:
            num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_weight = tf.pow(1.0 - y_pred, self.gamma)
        focal_loss = focal_weight * cross_entropy
        if self.alpha is not None:
            alpha_tensor = tf.constant(self.alpha, dtype=tf.float32)
            focal_loss = focal_loss * alpha_tensor
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update({
            'gamma': self.gamma,
            'alpha': self.alpha,
            'label_smoothing': self.label_smoothing
        })
        return config


def get_loss_function(class_weights=None):
    """Get appropriate loss function based on configuration."""
    if Config.USE_FOCAL_LOSS:
        if class_weights:
            alpha = [class_weights[i] for i in range(Config.NUM_CLASSES)]
        else:
            alpha = None
        print(f"Using Focal Loss (gamma={Config.FOCAL_GAMMA}, alpha={alpha})")
        return FocalLoss(gamma=Config.FOCAL_GAMMA, alpha=alpha, label_smoothing=Config.LABEL_SMOOTHING)
    else:
        print(f"Using Categorical Crossentropy (label_smoothing={Config.LABEL_SMOOTHING})")
        return keras.losses.CategoricalCrossentropy(label_smoothing=Config.LABEL_SMOOTHING)


print("Focal Loss implementation loaded!")

def apply_logit_adjustment(preds, class_counts, tau=1.0):
    """Apply logit adjustment to correct prior shift."""
    train_priors = class_counts / np.sum(class_counts)
    test_priors = np.ones_like(train_priors) / len(train_priors)  # Assume uniform test prior

    # Convert to logits if probabilities
    if np.all(preds >= 0) and np.all(preds <= 1):
        # Add epsilon to avoid log(0)
        preds = np.log(preds + 1e-10)

    # Apply adjustment: logit_adj = logit - log(train_prior) + log(test_prior)
    adjustment = np.log(test_priors + 1e-10) - np.log(train_priors + 1e-10)
    adjusted_logits = preds + adjustment * tau

    # Convert back to probabilities
    exp_logits = np.exp(adjusted_logits - np.max(adjusted_logits, axis=1, keepdims=True))
    adjusted_proba = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    return adjusted_proba

## 4. Dataset Download and Preparation

In [ ]:
def download_azh_dataset():
    """Download the AZH Wound Dataset from GitHub."""
    print("\n" + "=" * 70)
    print("DOWNLOADING AZH WOUND DATASET")
    print("=" * 70)
    repo_url = "https://github.com/uwm-bigdata/wound-classification-using-images-and-locations.git"
    if not os.path.exists(Config.REPO_DIR):
        print(f"Cloning repository...")
        !git clone {repo_url} {Config.REPO_DIR}
        print("Repository cloned!")
    else:
        print("Repository already exists.")
    print("\nRepository contents:")
    for item in os.listdir(Config.REPO_DIR):
        item_path = os.path.join(Config.REPO_DIR, item)
        size = os.path.getsize(item_path) if os.path.isfile(item_path) else "<dir>"
        print(f"  {item}: {size}")
    return Config.REPO_DIR


repo_dir = download_azh_dataset()

In [ ]:
def prepare_wound_dataset(zip_or_dir: str, output_root: str, split_name: str, copy: bool = True, ignore_bg: bool = True, dry_run: bool = False) -> pd.DataFrame:
    """Prepare the AZH wound dataset."""
    print(f"\n{'='*70}")
    print(f"PREPARING DATASET: {split_name.upper()}")
    print(f"{'='*70}")
    if zip_or_dir.endswith('.zip'):
        extract_dir = os.path.join(Config.RAW_DATA_DIR, split_name)
        if not dry_run:
            os.makedirs(extract_dir, exist_ok=True)
            print(f"Extracting {zip_or_dir}...")
            with zipfile.ZipFile(zip_or_dir, 'r') as zf:
                zf.extractall(extract_dir)
        source_dir = extract_dir
    else:
        source_dir = zip_or_dir
    if not dry_run:
        for subdir in ['Train', 'Test', split_name.capitalize(), split_name]:
            check_path = os.path.join(source_dir, subdir)
            if os.path.exists(check_path):
                source_dir = check_path
                break
    output_split_dir = os.path.join(output_root, split_name)
    for class_name in Config.CLASS_NAMES:
        if not dry_run:
            os.makedirs(os.path.join(output_split_dir, class_name), exist_ok=True)
    metadata = []
    for folder_code, class_name in Config.FOLDER_TO_CLASS.items():
        folder_path = os.path.join(source_dir, folder_code)
        if not os.path.exists(folder_path):
            continue
        image_files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            image_files.extend(glob.glob(os.path.join(folder_path, ext)))
        print(f"  {folder_code}/ ({class_name}): {len(image_files)} images")
        for img_path in image_files:
            filename = os.path.basename(img_path)
            orig_id = filename.split('_')[0] if '_' in filename else filename.split('.')[0]
            new_filename = f"{folder_code}_{filename}"
            dest_path = os.path.join(output_split_dir, class_name, new_filename)
            if not dry_run:
                shutil.copy2(img_path, dest_path)
            metadata.append({
                'split': split_name,
                'orig_folder': folder_code,
                'orig_id': orig_id,
                'orig_filename': filename,
                'new_filename': new_filename,
                'label': class_name,
                'filepath': dest_path
            })
    df_metadata = pd.DataFrame(metadata)
    print(f"\nTotal: {len(df_metadata)} images")
    if len(df_metadata) > 0:
        print(df_metadata['label'].value_counts().to_string())
    return df_metadata


def organize_full_dataset(repo_dir: str, output_root: str, dry_run: bool = False):
    """Organize the complete dataset."""
    print("\n" + "#" * 70)
    print("# ORGANIZING FULL AZH WOUND DATASET")
    print("#" * 70)
    train_zip = None
    test_zip = None
    for root, dirs, files in os.walk(repo_dir):
        for f in files:
            if f.lower() == 'train.zip':
                train_zip = os.path.join(root, f)
            elif f.lower() == 'test.zip':
                test_zip = os.path.join(root, f)
    print(f"Train zip: {train_zip}")
    print(f"Test zip: {test_zip}")
    train_metadata = pd.DataFrame()
    test_metadata = pd.DataFrame()
    if train_zip and os.path.exists(train_zip):
        train_metadata = prepare_wound_dataset(train_zip, output_root, 'train', dry_run=dry_run)
    if test_zip and os.path.exists(test_zip):
        test_metadata = prepare_wound_dataset(test_zip, output_root, 'test', dry_run=dry_run)
    all_metadata = pd.concat([train_metadata, test_metadata], ignore_index=True)
    if not dry_run and len(all_metadata) > 0:
        all_metadata.to_csv(os.path.join(output_root, 'dataset_metadata.csv'), index=False)
    print("\n" + "#" * 70)
    print("# DATASET ORGANIZATION COMPLETE")
    print("#" * 70)
    return train_metadata, test_metadata


train_metadata, test_metadata = organize_full_dataset(repo_dir=Config.REPO_DIR, output_root=Config.DATA_DIR, dry_run=False)

## 5. Advanced Data Generator with Mixup, CutMix and Oversampling

In [ ]:
class AdvancedDataGenerator(Sequence):
    """Advanced data generator with Mixup, CutMix, and oversampling."""

    def __init__(self, dataframe, batch_size, img_size, preprocess_func,
                 augment=False, mixup=False, cutmix=False, mixup_alpha=0.2,
                 oversample=False, shuffle=True):
        self.dataframe = dataframe.copy()
        self.batch_size = batch_size
        self.img_size = img_size
        self.preprocess_func = preprocess_func
        self.augment = augment
        self.mixup = mixup
        self.cutmix = cutmix
        self.mixup_alpha = mixup_alpha
        self.oversample = oversample
        self.shuffle = shuffle
        self.class_names = Config.CLASS_NAMES
        self.n_classes = len(self.class_names)
        self.label_to_idx = {name: idx for idx, name in enumerate(self.class_names)}

        # Gradient clipping por classe
        self.gradient_clip_norm = {'pressure': 0.5}

        if self.oversample:
            self._oversample_minority()
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def _oversample_minority(self):
        """Oversample minority classes to balance the dataset."""
        class_counts = self.dataframe['label'].value_counts()
        max_count = class_counts.max()
        print(f"\nOversampling to balance classes (target: {max_count} per class)")
        balanced_dfs = []
        for class_name in self.class_names:
            class_df = self.dataframe[self.dataframe['label'] == class_name]
            current_count = len(class_df)
            if current_count < max_count:
                # Aumentar oversampling para diabetic/surgical de 0.7→0.8
                if class_name == 'pressure':
                    oversample_factor = 0.6  # Mantém reduzido
                elif class_name in ['diabetic', 'surgical']:
                    oversample_factor = 0.8  # ERA 0.7
                else:
                    oversample_factor = 0.7  # venous (se aplicável)

                oversample_count = int((max_count - current_count) * oversample_factor)
                oversampled = class_df.sample(n=oversample_count, replace=True, random_state=SEED)
                class_df = pd.concat([class_df, oversampled], ignore_index=True)
                print(f"  {class_name}: {current_count} → {len(class_df)} (factor={oversample_factor})")
            else:
                print(f"  {class_name}: {current_count} (unchanged)")
            balanced_dfs.append(class_df)
        self.dataframe = pd.concat(balanced_dfs, ignore_index=True)
        print(f"  Total after oversampling: {len(self.dataframe)}")

    def __len__(self):
        return int(np.ceil(len(self.dataframe) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_df = self.dataframe.iloc[batch_indices]
        images = []
        labels = []
        for _, row in batch_df.iterrows():
            img = self._load_and_preprocess(row['filepath'])
            label = self.label_to_idx[row['label']]
            if self.augment:
                img = self._augment_image(img)
            images.append(img)
            labels.append(label)
        images = np.array(images)
        labels = to_categorical(labels, num_classes=self.n_classes)
        if self.mixup and self.augment:
            images, labels = self._mixup_batch(images, labels)
        elif self.cutmix and self.augment:
            images, labels = self._cutmix_batch(images, labels)
        return images, labels

    def _load_and_preprocess(self, filepath):
        """Load and preprocess image."""
        img = cv2.imread(filepath)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = img.astype(np.float32)
        img = self.preprocess_func(img)
        return img

    def _augment_image(self, img):
        """Apply augmentation to image."""
        if Config.HORIZONTAL_FLIP and np.random.random() > 0.5:
            img = np.fliplr(img)
        if Config.ROTATION_RANGE > 0:
            angle = np.random.uniform(-Config.ROTATION_RANGE, Config.ROTATION_RANGE)
            M = cv2.getRotationMatrix2D((self.img_size/2, self.img_size/2), angle, 1)
            img = cv2.warpAffine(img, M, (self.img_size, self.img_size))
        if Config.BRIGHTNESS_RANGE:
            factor = np.random.uniform(Config.BRIGHTNESS_RANGE[0], Config.BRIGHTNESS_RANGE[1])
            img = img * factor
            img = np.clip(img, -1, 1) if img.min() < 0 else np.clip(img, 0, 255)
        return img

    def _mixup_batch(self, images, labels):
        """Apply mixup augmentation to batch."""
        batch_size = len(images)
        indices = np.random.permutation(batch_size)
        lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
        mixed_images = lam * images + (1 - lam) * images[indices]
        mixed_labels = lam * labels + (1 - lam) * labels[indices]
        return mixed_images, mixed_labels

    def _cutmix_batch(self, images, labels):
        """Apply CutMix augmentation."""
        batch_size = len(images)
        indices = np.random.permutation(batch_size)
        lam = np.random.beta(Config.CUTMIX_ALPHA, Config.CUTMIX_ALPHA)
        W, H = self.img_size, self.img_size
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        mixed_images = images.copy()
        mixed_images[:, bby1:bby2, bbx1:bbx2, :] = images[indices, bby1:bby2, bbx1:bbx2, :]
        lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (W * H))
        mixed_labels = lam * labels + (1 - lam) * labels[indices]
        return mixed_images, mixed_labels

    def on_epoch_end(self):
        """Shuffle indices at end of epoch."""
        if self.shuffle:
            np.random.shuffle(self.indices)

    @property
    def samples(self):
        return len(self.dataframe)

    @property
    def classes(self):
        return self.dataframe['label'].map(self.label_to_idx).values

    @property
    def class_indices(self):
        return self.label_to_idx

    @property
    def filepaths(self):
        return self.dataframe['filepath'].tolist()


print("AdvancedDataGenerator loaded!")

In [ ]:
def compute_class_weights_from_metadata(metadata_df):
    """Compute class weights from metadata."""
    labels = metadata_df['label'].values
    label_encoder = {name: idx for idx, name in enumerate(Config.CLASS_NAMES)}
    encoded_labels = [label_encoder[l] for l in labels]
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(encoded_labels), y=encoded_labels)
    class_weight_dict = {i: w for i, w in enumerate(weights)}
    print("\nClass distribution and weights:")
    for idx, class_name in enumerate(Config.CLASS_NAMES):
        count = sum(1 for l in labels if l == class_name)
        print(f"  {class_name}: {count} samples, weight={class_weight_dict[idx]:.3f}")
    # Ajuste pontual para rebalanceamento da classe pressure
    # Reduzido de 2.5 para 1.5 (estava causando overprediction)
    class_weight_dict[Config.CLASS_NAMES.index('pressure')] = 1.2
    print(f"\\n Rebalanceamento LEVE: class_weight['pressure'] = 1.2 (era 2.5)")
    return class_weight_dict


def create_advanced_generators(model_type='mobilenet'):
    """Create advanced data generators with all optimizations."""
    print("\n" + "=" * 70)
    print(f"CREATING ADVANCED GENERATORS FOR {model_type.upper()}")
    print("=" * 70)
    if model_type == 'mobilenet':
        preprocess_func = mobilenet_preprocess
        print("Using MobileNetV2 preprocessing")
    elif model_type == 'efficientnet':
        preprocess_func = efficientnet_preprocess
        print("Using EfficientNet preprocessing")
    elif model_type == 'densenet':
        preprocess_func = densenet_preprocess
        print("Using DenseNet121 preprocessing")
    train_dir = os.path.join(Config.DATA_DIR, 'train')
    test_dir = os.path.join(Config.DATA_DIR, 'test')
    train_data = []
    for class_name in Config.CLASS_NAMES:
        class_dir = os.path.join(train_dir, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    train_data.append({'filepath': os.path.join(class_dir, f), 'label': class_name})
    test_data = []
    for class_name in Config.CLASS_NAMES:
        class_dir = os.path.join(test_dir, class_name)
        if os.path.exists(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    test_data.append({'filepath': os.path.join(class_dir, f), 'label': class_name})
    train_df = pd.DataFrame(train_data)
    test_df = pd.DataFrame(test_data)
    train_df_split, val_df = train_test_split(train_df, test_size=Config.VAL_SPLIT, stratify=train_df['label'], random_state=SEED)
    print(f"\nTraining samples: {len(train_df_split)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Test samples: {len(test_df)}")
    class_weights = compute_class_weights_from_metadata(train_df_split)
    train_gen = AdvancedDataGenerator(dataframe=train_df_split, batch_size=Config.BATCH_SIZE, img_size=Config.IMG_SIZE, preprocess_func=preprocess_func, augment=True, mixup=Config.USE_MIXUP, cutmix=Config.USE_CUTMIX, mixup_alpha=Config.MIXUP_ALPHA, oversample=Config.OVERSAMPLE_MINORITY, shuffle=True)
    val_gen = AdvancedDataGenerator(dataframe=val_df, batch_size=Config.BATCH_SIZE, img_size=Config.IMG_SIZE, preprocess_func=preprocess_func, augment=False, mixup=False, oversample=False, shuffle=False)
    test_gen = AdvancedDataGenerator(dataframe=test_df, batch_size=Config.BATCH_SIZE, img_size=Config.IMG_SIZE, preprocess_func=preprocess_func, augment=False, mixup=False, oversample=False, shuffle=False)
    print(f"\nAdvanced features enabled:")
    print(f"  - Mixup: {Config.USE_MIXUP}")
    print(f"  - CutMix: {Config.USE_CUTMIX}")
    print(f"  - Oversampling: {Config.OVERSAMPLE_MINORITY}")
    print(f"  - Training batches per epoch: {len(train_gen)}")
    return train_gen, val_gen, test_gen, class_weights

## 6. Model Architecture with Advanced Features

In [ ]:
def build_advanced_model(base_model_type='mobilenet'):
    """Build model with advanced architecture."""
    print("\n" + "=" * 70)
    print(f"BUILDING ADVANCED {base_model_type.upper()} MODEL")
    print("=" * 70)
    if base_model_type == 'mobilenet':
        base_model = MobileNetV2(input_shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3), include_top=False, weights='imagenet')
    elif base_model_type == 'efficientnet':
        base_model = EfficientNetB0(input_shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3), include_top=False, weights='imagenet')
    elif base_model_type == 'densenet':
        base_model = DenseNet121(input_shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = keras.Input(shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(Config.DENSE_UNITS_1, kernel_regularizer=keras.regularizers.l2(Config.L2_REGULARIZATION), kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(Config.DROPOUT_RATE)(x)
    x = layers.Dense(Config.DENSE_UNITS_2, kernel_regularizer=keras.regularizers.l2(Config.L2_REGULARIZATION), kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(Config.DROPOUT_RATE * 0.7)(x)
    outputs = layers.Dense(Config.NUM_CLASSES, activation='softmax', kernel_initializer='glorot_uniform')(x)
    model = keras.Model(inputs, outputs, name=f'WoundClassifier_{base_model_type}_v6')
    print(f"\nModel: {base_model_type} (Advanced v6)")
    print(f"Total parameters: {model.count_params():,}")
    print(f"Trainable parameters (Phase 1): {sum([np.prod(w.shape) for w in model.trainable_weights]):,}")
    return model, base_model

## 7. Advanced Training with Warm-up and Cosine Annealing

In [ ]:
# Decorar classes para serialização Keras (compatibilidade com TF 2.x)
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable()
class WarmUpCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    """Learning rate schedule with warm-up and cosine decay."""

    def __init__(self, initial_lr, warmup_steps, total_steps):
        super().__init__()
        self.initial_lr = initial_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.initial_lr * (step / self.warmup_steps)
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = self.initial_lr * 0.5 * (1 + tf.cos(np.pi * progress))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'initial_lr': self.initial_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps': self.total_steps
        }

# Decorar FocalLoss também para serialização
@register_keras_serializable()
class FocalLoss(keras.losses.Loss):
    """Focal Loss for handling class imbalance."""

    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        if self.label_smoothing > 0:
            num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + (self.label_smoothing / num_classes)
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_weight = tf.pow(1.0 - y_pred, self.gamma)
        focal_loss = focal_weight * cross_entropy
        if self.alpha is not None:
            alpha_tensor = tf.constant(self.alpha, dtype=tf.float32)
            focal_loss = focal_loss * alpha_tensor
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update({
            'gamma': self.gamma,
            'alpha': self.alpha,
            'label_smoothing': self.label_smoothing
        })
        return config

def get_advanced_callbacks(model_name):
    """Get callback list for training (simplificado)."""
    callbacks_list = []

    # Early stopping
    callbacks_list.append(callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=Config.EARLY_STOPPING_PATIENCE,  # Remova o * patience_mult
        restore_best_weights=True,
        verbose=1
    ))

    # Model checkpoint
    callbacks_list.append(callbacks.ModelCheckpoint(
        os.path.join(Config.MODEL_SAVE_DIR, f'{model_name}_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ))

    return callbacks_list

@register_keras_serializable()
class DiscriminativeLRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    """Discriminative learning rate for different layer groups."""

    def __init__(self, backbone_lr, head_lr, total_steps):
        super().__init__()
        self.backbone_lr = backbone_lr
        self.head_lr = head_lr
        self.total_steps = total_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        backbone_factor = tf.where(step < self.total_steps * 0.5, 1.0, 0.5)
        return [self.backbone_lr * backbone_factor, self.head_lr]

    def get_config(self):
        return {
            'backbone_lr': self.backbone_lr,
            'head_lr': self.head_lr,
            'total_steps': self.total_steps
        }

# Atualizar a função de carregamento do modelo
def train_advanced_model(model, base_model, train_gen, val_gen, test_gen, class_weights, model_name):
    """Treina apenas Phase 1 (feature extraction) - SEM fine-tuning."""
    print("\n" + "=" * 70)
    print(f"TRAINING {model_name.upper()} (PHASE 1 ONLY)")
    print("=" * 70)

    loss_fn = get_loss_function(class_weights)

    # Apenas Phase 1: Treinar cabeça de classificação
    print(f"\n--- PHASE 1: Training classification head ---")
    print(f"Epochs: {Config.EPOCHS_PHASE1}")
    print(f"Learning rate: {Config.LEARNING_RATE}")

    steps_per_epoch = len(train_gen)
    warmup_steps = Config.WARMUP_EPOCHS * steps_per_epoch
    total_steps = Config.EPOCHS_PHASE1 * steps_per_epoch
    lr_schedule = WarmUpCosineDecay(
        initial_lr=Config.LEARNING_RATE,
        warmup_steps=warmup_steps,
        total_steps=total_steps
    )

    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr_schedule),
        loss=loss_fn,
        metrics=['accuracy']
    )

    history = model.fit(
        train_gen,
        epochs=Config.EPOCHS_PHASE1,
        validation_data=val_gen,
        callbacks=get_advanced_callbacks(f'{model_name}_phase1'),
        class_weight=class_weights,
        verbose=1
    )

    # Salvar modelo final (que é apenas o Phase 1)
    final_path = os.path.join(Config.MODEL_SAVE_DIR, f'{model_name}_final.keras')
    model.save(final_path)
    print(f"\n✓ Model saved: {final_path}")

    return history, model

## 8. Test-Time Augmentation (TTA) and Uncertainty

In [ ]:
def predict_with_uncertainty(model, test_gen, preprocess_func, n_augments=10):
    """Make predictions with TTA and compute uncertainty metrics."""
    print(f"\nApplying Test-Time Augmentation ({n_augments} augmentations)...")
    all_predictions = []
    test_gen_original = AdvancedDataGenerator(dataframe=pd.DataFrame({'filepath': test_gen.filepaths, 'label': [Config.CLASS_NAMES[c] for c in test_gen.classes]}), batch_size=Config.BATCH_SIZE, img_size=Config.IMG_SIZE, preprocess_func=preprocess_func, augment=False, shuffle=False)
    preds_original = model.predict(test_gen_original, verbose=0)
    all_predictions.append(preds_original)
    for i in range(n_augments - 1):
        test_gen_aug = AdvancedDataGenerator(dataframe=pd.DataFrame({'filepath': test_gen.filepaths, 'label': [Config.CLASS_NAMES[c] for c in test_gen.classes]}), batch_size=Config.BATCH_SIZE, img_size=Config.IMG_SIZE, preprocess_func=preprocess_func, augment=True, shuffle=False)
        preds_aug = model.predict(test_gen_aug, verbose=0)
        all_predictions.append(preds_aug)
    avg_predictions = np.mean(all_predictions, axis=0)
    predictive_entropy = -np.sum(avg_predictions * np.log(avg_predictions + 1e-10), axis=1)
    variance = np.var(all_predictions, axis=0)
    avg_variance = np.mean(variance, axis=1)
    print(f"TTA complete. Averaged {n_augments} predictions.")
    print(f"Mean predictive entropy: {np.mean(predictive_entropy):.4f}")
    print(f"Mean variance: {np.mean(avg_variance):.4f}")
    return avg_predictions, predictive_entropy, avg_variance

## 9. Advanced Evaluation Functions

In [ ]:
def plot_training_history(history, model_name):
    """Plot training curves with early stopping marker."""
    # Se receber um objeto History, extrair o dicionário
    if hasattr(history, 'history'):
        history = history.history

    # Verificar as chaves disponíveis (compatibilidade com diferentes versões do Keras)
    val_acc_key = 'val_accuracy' if 'val_accuracy' in history else 'val_acc'
    acc_key = 'accuracy' if 'accuracy' in history else 'acc'
    val_loss_key = 'val_loss'
    loss_key = 'loss'

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    best_epoch = np.argmax(history[val_acc_key])

    # Plot loss
    axes[0].plot(history[loss_key], label='Training Loss', linewidth=2)
    axes[0].plot(history[val_loss_key], label='Validation Loss', linewidth=2)
    axes[0].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch: {best_epoch}')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f"{model_name} - Loss Curves")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot accuracy
    axes[1].plot(history[acc_key], label='Training Accuracy', linewidth=2)
    axes[1].plot(history[val_acc_key], label='Validation Accuracy', linewidth=2)
    axes[1].axvline(x=best_epoch, color='red', linestyle='--', alpha=0.5, label=f'Best Epoch: {best_epoch}')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f"{model_name} - Accuracy Curves")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(Config.RESULTS_DIR, exist_ok=True)
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{model_name}_training_curves.png'), dpi=150)
    plt.show()


def compute_ece(y_true, y_pred_proba, n_bins=15):
    """Calculate Expected Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    ece = 0.0
    calibration_data = []
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        confidences = np.max(y_pred_proba, axis=1)
        in_bin = (confidences >= bin_lower) & (confidences < (bin_uppers[-1] if bin_upper == 1.0 else bin_upper))
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy = np.mean(np.argmax(y_pred_proba[in_bin], axis=1) == np.array(y_true)[in_bin])
            avg_confidence = np.mean(confidences[in_bin])
            ece += prop_in_bin * abs(accuracy - avg_confidence)
            calibration_data.append({'bin': f"[{bin_lower:.2f}, {bin_upper:.2f})", 'accuracy': accuracy, 'confidence': avg_confidence, 'samples': np.sum(in_bin)})
    return ece, pd.DataFrame(calibration_data)


def plot_reliability_diagram(results, model_name):
    """Plot reliability diagram for calibration assessment."""
    ece, cal_data = compute_ece(results['y_true'], results['y_pred_proba'])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    ax1.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax1.plot(cal_data['confidence'], cal_data['accuracy'], 'o-', label=f'ECE = {ece:.4f}', linewidth=2)
    ax1.set_xlabel('Confidence')
    ax1.set_ylabel('Accuracy')
    ax1.set_title(f'{model_name} - Reliability Diagram')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    confidences = np.max(results['y_pred_proba'], axis=1)
    ax2.hist(confidences, bins=15, edgecolor='black', alpha=0.7)
    ax2.set_xlabel('Confidence')
    ax2.set_ylabel('Count')
    ax2.set_title('Confidence Distribution')
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{model_name}_reliability.png'), dpi=150)
    plt.show()
    print(f"Expected Calibration Error: {ece:.4f}")
    return ece


def evaluate_model_advanced(model, test_gen, preprocess_func, model_name, use_tta=True, return_probs=False):
    """Evaluate model with TTA and comprehensive metrics."""
    print("\n" + "=" * 70)
    print(f"EVALUATING {model_name.upper()}")
    print("=" * 70)

    if use_tta and Config.USE_TTA:
        y_pred_proba, uncertainty, variance = predict_with_uncertainty(model, test_gen, preprocess_func, n_augments=Config.TTA_STEPS)
    else:
        y_pred_proba = model.predict(test_gen, verbose=1)
        uncertainty = None
        variance = None

        # Logit adjustment condicional (pode causar sink class se ativo)
    if Config.APPLY_LOGIT_ADJUSTMENT:
        print(f"⚠️  Logit adjustment ATIVO (tau={Config.LOGIT_ADJUSTMENT_TAU}) - pode amplificar sink class")
        train_counts = np.bincount(test_gen.classes)
        y_pred_proba = apply_logit_adjustment(y_pred_proba, train_counts, tau=Config.LOGIT_ADJUSTMENT_TAU)
    else:
        print(f"✓ Logit adjustment DESATIVADO (use Config.APPLY_LOGIT_ADJUSTMENT=True para ativar)")

    y_pred = np.argmax(y_pred_proba, axis=1)
    y_true = test_gen.classes
    class_names = list(test_gen.class_indices.keys())
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    ece = compute_ece(y_true, y_pred_proba)[0]
    ci_low, ci_upp = proportion_confint(accuracy * len(y_true), len(y_true), alpha=0.05, method='wilson')
    print("\nClassification Report:")
    print("-" * 60)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
    # FOCAR EM MACRO F1 (Weighted esconde problemas de classes minoritárias)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1  # já calculado
    print("\\nSummary Metrics (FOCO EM MACRO F1):")
    print("-" * 60)
    print(f"  Accuracy:      {accuracy:.4f} ({accuracy*100:.2f}%) [95% CI: {ci_low:.1%}, {ci_upp:.1%}]")
    print(f"  Macro F1:      {macro_f1:.4f} ⚠️ ATENÇÃO: Métrica principal para classes minoritárias")
    print(f"  Weighted F1:   {weighted_f1:.4f} (pode esconder problemas de pressure/surgical)")
    print(f"  Macro-Weighted Gap: {abs(macro_f1-weighted_f1):.4f} (gap > 0.02 = viés severo)")
    print(f"  Precision:     {precision:.4f}")
    print(f"  Recall:        {recall:.4f}")
    print(f"  ECE:           {ece:.4f}")
    if use_tta and Config.USE_TTA:
        print(f"  (with TTA: {Config.TTA_STEPS} augmentations)")
        print(f"  Mean uncertainty: {np.mean(uncertainty):.4f}")
        print(f"  Mean variance: {np.mean(variance):.4f}")
    results = {
        'model_name': model_name, 'accuracy': accuracy, 'ci_low': ci_low, 'ci_upp': ci_upp,
        'precision': precision, 'recall': recall, 'f1_score': f1, 'ece': ece,
        'y_true': y_true, 'y_pred': y_pred, 'y_pred_proba': y_pred_proba,
        'class_names': class_names, 'uncertainty': uncertainty, 'variance': variance
    }

    # Auto-optimize thresholds if enabled
    if Config.OPTIMIZE_THRESHOLDS:
        print(f"\n[Auto-optimizing thresholds for {model_name}]")
        thresholds = optimize_class_thresholds(results, metric=Config.THRESHOLD_METRIC)
        results_thr = apply_thresholds_and_evaluate(results, thresholds)
        return results, results_thr

    return results

In [ ]:
def plot_confusion_matrix(results, model_name):
    """Plot confusion matrix with counts and percentages."""
    cm = confusion_matrix(results['y_true'], results['y_pred'])
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=results['class_names'], yticklabels=results['class_names'], ax=axes[0])
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].set_title(f'{model_name} - Confusion Matrix (Counts)')
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', xticklabels=results['class_names'], yticklabels=results['class_names'], ax=axes[1])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].set_title(f'{model_name} - Confusion Matrix (Normalized)')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{model_name}_confusion_matrix.png'), dpi=150)
    plt.show()
    print("\nClass-wise Recall with 95% CI:")
    for i, class_name in enumerate(results['class_names']):
        n_correct = cm[i, i]
        n_total = cm[i, :].sum()
        ci_low, ci_upp = proportion_confint(n_correct, n_total, alpha=0.05, method='wilson')
        print(f"  {class_name}: {n_correct/n_total:.1%} [{ci_low:.1%}, {ci_upp:.1%}]")
        # Sink class diagnosis
    print("\n" + "="*70)
    print(f"SINK CLASS DIAGNOSIS: {results['model_name']}")
    print("="*70)

    pressure_idx = results['class_names'].index('pressure')
    total_errors = np.sum(cm) - np.trace(cm)
    fp_pressure = np.sum(cm[:, pressure_idx]) - cm[pressure_idx, pressure_idx]
    print(f"\nTotal errors: {total_errors}")
    print(f"False Positives for pressure: {fp_pressure} ({fp_pressure/total_errors:.1%} of all errors)")

    print("\nBreakdown by true class:")
    for i, class_name in enumerate(results['class_names']):
        if i == pressure_idx:
            continue
        errors_from_class = np.sum(cm[i, :]) - cm[i, i]
        fp_to_pressure = cm[i, pressure_idx]
        pct_sink = fp_to_pressure / errors_from_class if errors_from_class > 0 else 0
        print(f"  {class_name}: {errors_from_class} errors → {fp_to_pressure} viram pressure ({pct_sink:.1%})")
    return cm

def diagnose_pressure_sink(results):
    """Exibe diagnóstico completo do sink class como exigido pelo orientador"""
    cm = confusion_matrix(results['y_true'], results['y_pred'])
    class_names = results['class_names']

    print("\n" + "="*70)
    print(f"SINK CLASS DIAGNOSIS: {results['model_name']}")
    print("="*70)

    # Total errors
    total_errors = np.sum(cm) - np.trace(cm)
    print(f"\nTotal errors: {total_errors}")

    # Pressure FP breakdown
    pressure_idx = class_names.index('pressure')
    fp_pressure = np.sum(cm[:, pressure_idx]) - cm[pressure_idx, pressure_idx]
    print(f"False Positives for pressure: {fp_pressure} ({fp_pressure/total_errors:.1%} of all errors)")

    # Per-origin breakdown
    print("\nBreakdown by true class:")
    sink_data = []
    for i, class_name in enumerate(class_names):
        if i == pressure_idx:
            continue
        errors_from_class = np.sum(cm[i, :]) - cm[i, i]
        fp_to_pressure = cm[i, pressure_idx]
        pct_sink = fp_to_pressure / errors_from_class if errors_from_class > 0 else 0
        print(f"  {class_name}: {errors_from_class} errors → {fp_to_pressure} viram pressure ({pct_sink:.1%})")
        sink_data.append({'Class': class_name, 'Errors': errors_from_class, 'FP_to_Pressure': fp_to_pressure, 'Pct_Sink': pct_sink})

    return pd.DataFrame(sink_data)


def compare_fp_pressure(results_list):
    """Compara FP de pressure entre modelos explicitamente"""
    print("\n" + "="*70)
    print("FALSE POSITIVE COMPARISON - PRESSURE CLASS")
    print("="*70)

    fp_data = []
    for r in results_list:
        cm = confusion_matrix(r['y_true'], r['y_pred'])
        pressure_idx = r['class_names'].index('pressure')
        fp = np.sum(cm[:, pressure_idx]) - cm[pressure_idx, pressure_idx]
        fp_data.append({'Model': r['model_name'], 'FP_Pressure': fp})

    df = pd.DataFrame(fp_data)
    print(df.to_string(index=False))

    if len(df) == 2:
        reduction = df.loc[0, 'FP_Pressure'] - df.loc[1, 'FP_Pressure']
        pct_reduction = reduction / df.loc[0, 'FP_Pressure'] * 100
        print(f"\n✓ Redução de FP: {reduction} amostras ({pct_reduction:.1f}%)")

    return df

def plot_precision_recall_curves(results, model_name):
    """Plot precision-recall curves."""
    fig, ax = plt.subplots(figsize=(10, 8))
    y_true_binary = to_categorical(results['y_true'], num_classes=len(results['class_names']))
    for i, class_name in enumerate(results['class_names']):
        precision, recall, _ = precision_recall_curve(y_true_binary[:, i], results['y_pred_proba'][:, i])
        pr_auc = auc(recall, precision)
        ax.plot(recall, precision, linewidth=2, label=f'{class_name} (AP = {pr_auc:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{model_name} - Precision-Recall Curves')
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{model_name}_pr_curves.png'), dpi=150)
    plt.show()


def plot_roc_curves(results, model_name):
    """Plot ROC curves with per-class AUC."""
    n_classes = len(results['class_names'])
    y_true_binary = to_categorical(results['y_true'], num_classes=n_classes)
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, n_classes))
    auc_values = {}
    for i, (class_name, color) in enumerate(zip(results['class_names'], colors)):
        fpr, tpr, _ = roc_curve(y_true_binary[:, i], results['y_pred_proba'][:, i])
        roc_auc = auc(fpr, tpr)
        auc_values[class_name] = roc_auc
        ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{class_name} (AUC = {roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{model_name} - ROC Curves')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{model_name}_roc_curves.png'), dpi=150)
    plt.show()
    return auc_values

## 10. Statistical Comparison Tests

In [ ]:
def compare_models_mcnemar(results_a, results_b, alpha=0.05):
    """McNemar's test for paired classifiers."""
    print("\n" + "=" * 70)
    print("MCNEMAR'S TEST")
    print("=" * 70)
    y_true = results_a['y_true']
    y_pred_a = results_a['y_pred']
    y_pred_b = results_b['y_pred']
    tb = np.zeros((2, 2), dtype=np.int64)
    for i in range(len(y_true)):
        a_correct = y_pred_a[i] == y_true[i]
        b_correct = y_pred_b[i] == y_true[i]
        if a_correct and b_correct:
            tb[0, 0] += 1
        elif a_correct and not b_correct:
            tb[0, 1] += 1
        elif not a_correct and b_correct:
            tb[1, 0] += 1
        else:
            tb[1, 1] += 1
    print(f"\nContingency Table:")
    print(f"                {results_b['model_name']} Correct | {results_b['model_name']} Wrong")
    print(f"{results_a['model_name']} Correct | {tb[0,0]:4d}             | {tb[0,1]:4d}")
    print(f"{results_a['model_name']} Wrong   | {tb[1,0]:4d}             | {tb[1,1]:4d}")
    result = mcnemar(tb, exact=False)
    print(f"\nMcNemar Test p-value: {result.pvalue:.6f}")
    print(f"Significant (p < {alpha}): {'Sim' if result.pvalue < alpha else 'Nao'}")
    if result.pvalue < alpha:
        if tb[1, 0] > tb[0, 1]:
            print(f"Conclusion: {results_b['model_name']} is significantly better")
        else:
            print(f"Conclusion: {results_a['model_name']} is significantly better")
    else:
        print(f"Conclusion: No significant difference between models")
    return result, tb


def ensemble_predictions(models_results, weights=None, use_uncertainty=True):
    """Combine predictions from multiple models with optional uncertainty weighting."""
    print("\n" + "=" * 70)
    print("ENSEMBLE PREDICTIONS")
    print("=" * 70)
    n_models = len(models_results)
    if weights is None:
        weights = [1.0 / n_models] * n_models
    print(f"\nCombining {n_models} models with weights: {weights}")
    ensemble_proba = np.zeros_like(models_results[0]['y_pred_proba'])
    for i, (result, weight) in enumerate(zip(models_results, weights)):
        print(f"- {result['model_name']}: weight={weight:.3f}, accuracy={result['accuracy']:.4f}, ECE={result['ece']:.4f}")
        ensemble_proba += weight * result['y_pred_proba']
    ensemble_pred = np.argmax(ensemble_proba, axis=1)
    y_true = models_results[0]['y_true']
    class_names = models_results[0]['class_names']
    accuracy = accuracy_score(y_true, ensemble_pred)
    precision = precision_score(y_true, ensemble_pred, average='weighted')
    recall = recall_score(y_true, ensemble_pred, average='weighted')
    f1 = f1_score(y_true, ensemble_pred, average='weighted')
    ece = compute_ece(y_true, ensemble_proba)[0]
    ci_low, ci_upp = proportion_confint(accuracy * len(y_true), len(y_true), alpha=0.05, method='wilson')
    print("\n" + "=" * 70)
    print("ENSEMBLE RESULTS")
    print("=" * 70)
    print(classification_report(y_true, ensemble_pred, target_names=class_names, digits=4))
    print("\nEnsemble Metrics:")
    print("-" * 60)
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%) [95% CI: {ci_low:.1%}, {ci_upp:.1%}]")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ECE:       {ece:.4f}")
    return {'model_name': 'Ensemble', 'accuracy': accuracy, 'ci_low': ci_low, 'ci_upp': ci_upp, 'precision': precision, 'recall': recall, 'f1_score': f1, 'ece': ece, 'y_true': y_true, 'y_pred': ensemble_pred, 'y_pred_proba': ensemble_proba, 'class_names': class_names, 'weights': weights}

## 11. Qualitative Analysis - Visualization

In [ ]:
def visualize_misclassifications(results, test_gen, n_examples=9, confidence_threshold=None):
    """Show misclassified examples with high confidence."""
    if confidence_threshold is None:
        confidence_threshold = Config.VISUALIZATION_CONFIDENCE_THRESHOLD
    errors = np.where(results['y_pred'] != results['y_true'])[0]
    high_conf_errors = [idx for idx in errors if np.max(results['y_pred_proba'][idx]) > confidence_threshold]
    if len(high_conf_errors) == 0:
        print("No high-confidence errors to visualize")
        return
    n_plot = min(n_examples, len(high_conf_errors))
    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.ravel()
    for i, idx in enumerate(high_conf_errors[:n_plot]):
        img_path = test_gen.filepaths[idx]
        img = load_img(img_path, target_size=(224, 224))
        true_label = results['class_names'][results['y_true'][idx]]
        pred_label = results['class_names'][results['y_pred'][idx]]
        confidence = np.max(results['y_pred_proba'][idx])
        axes[i].imshow(img)
        axes[i].set_title(f'True: {true_label}\\nPred: {pred_label}\\nConfidence: {confidence:.2%}', fontsize=10, color='darkred', fontweight='bold')
        axes[i].axis('off')
    for i in range(len(high_conf_errors[:n_plot]), 9):
        fig.delaxes(axes[i])
    plt.suptitle(f"{results['model_name']} - Misclassifications (confidence > {confidence_threshold})", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{results["model_name"]}_misclassifications.png'), dpi=150)
    plt.show()


def generate_gradcam(model, img_array, layer_name):
    """Generate Grad-CAM heatmap for interpretability."""
    grad_model = Model([model.inputs], [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, np.argmax(predictions[0])]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = np.maximum(heatmap, 0)
    heatmap /= np.max(heatmap) if np.max(heatmap) > 0 else 1
    return heatmap, predictions[0]


def visualize_gradcam_for_class(results, test_gen, model, preprocess_func, target_class, layer_name, n_examples=5):
    """Visualize Grad-CAM for correctly classified examples of a class."""
    class_idx = results['class_names'].index(target_class)
    correct = np.where((results['y_pred'] == results['y_true']) & (results['y_true'] == class_idx))[0]
    if len(correct) == 0:
        print(f"No correct examples found for {target_class}")
        return
    n_plot = min(n_examples, len(correct))
    fig, axes = plt.subplots(1, n_plot, figsize=(5 * n_plot, 5))
    if n_plot == 1:
        axes = [axes]
    for i, idx in enumerate(correct[:n_plot]):
        img_path = test_gen.filepaths[idx]
        img = load_img(img_path, target_size=(224, 224))
        img_array = np.expand_dims(img_to_array(img), axis=0)
        img_array = preprocess_func(img_array)
        heatmap, predictions = generate_gradcam(model, img_array, layer_name)
        axes[i].imshow(img)
        axes[i].imshow(cv2.resize(heatmap, (224, 224)), cmap='jet', alpha=0.4)
        axes[i].set_title(f'{target_class}\\nConf: {np.max(predictions):.2%}', fontsize=10, fontweight='bold')
        axes[i].axis('off')
    plt.suptitle(f'Grad-CAM - {results["model_name"]} - {target_class}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.RESULTS_DIR, f'{results["model_name"]}_gradcam_{target_class}.png'), dpi=150)
    plt.show()

## 12. Analysis Functions for Corrections

In [ ]:
def analyze_class_distribution_shift(results_list):
    """Analyze distribution shift between true and predicted classes."""
    print("\n" + "=" * 70)
    print("CLASS DISTRIBUTION SHIFT ANALYSIS")
    print("=" * 70)
    true_counts = np.bincount(results_list[0]['y_true'])
    true_dist = true_counts / len(results_list[0]['y_true'])
    print("\nTrue Test Distribution:")
    for i, class_name in enumerate(Config.CLASS_NAMES):
        print(f"  {class_name}: {true_counts[i]} ({true_dist[i]:.1%})")

    shift_data = []
    for results in results_list:
        pred_counts = np.bincount(results['y_pred'])
        pred_dist = pred_counts / len(results['y_pred'])
        print(f"\nPredicted Distribution ({results['model_name']}):")
        for i, class_name in enumerate(Config.CLASS_NAMES):
            shift = pred_dist[i] - true_dist[i]
            print(f"  {class_name}: {pred_counts[i]} ({pred_dist[i]:.1%}) [Shift: {shift:+.1%}]")
            shift_data.append({'Model': results['model_name'], 'Class': class_name, 'True_Prop': true_dist[i], 'Pred_Prop': pred_dist[i], 'Shift': shift})
    return pd.DataFrame(shift_data)


def analyze_false_positives_sink_class(results, sink_class='pressure'):
    """Analyze false positives with focus on sink class."""
    print("\n" + "=" * 70)
    print(f"FALSE POSITIVE ANALYSIS - SINK CLASS: {sink_class.upper()}")
    print("=" * 70)
    cm = confusion_matrix(results['y_true'], results['y_pred'])
    class_names = results['class_names']
    sink_idx = class_names.index(sink_class)
    total_fp = np.sum(cm[:, sink_idx]) - cm[sink_idx, sink_idx]
    total_errors = np.sum(cm) - np.trace(cm)
    print(f"\nTotal errors: {total_errors}")
    print(f"False Positives for {sink_class}: {total_fp} ({total_fp/total_errors:.1%} of all errors)")
    print(f"\nBreakdown by true class:")
    fp_data = []
    for i, class_name in enumerate(class_names):
        if i != sink_idx:
            fp_count = cm[i, sink_idx]
            fn_count = np.sum(cm[i, :]) - cm[i, i]
            if fn_count > 0:
                fp_percentage = fp_count / fn_count
                print(f"  {class_name} → {sink_class}: {fp_count} ({fp_percentage:.1%} of {class_name} errors)")
                fp_data.append({'True_Class': class_name, 'FP_to_Sink': fp_count, 'Pct_of_Errors': fp_percentage})
    return pd.DataFrame(fp_data)


def analyze_directional_confusion(results, threshold=0.05):
    """Análise detalhada de assimetria com thresholds ajustáveis"""
    cm = confusion_matrix(results['y_true'], results['y_pred'])
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    class_names = results['class_names']

    print("\n" + "="*70)
    print(f"DIRECTIONAL CONFUSION ASYMMETRY - {results['model_name']}")
    print("="*70)

    asymmetry_data = []
    for i in range(len(class_names)):
        for j in range(i + 1, len(class_names)):
            a_to_b = cm_normalized[i, j]
            b_to_a = cm_normalized[j, i]
            asymmetry = abs(a_to_b - b_to_a)

            # Mostra TODOS os pares com assimetria significativa
            if asymmetry > threshold or a_to_b > 0.1 or b_to_a > 0.1:
                print(f"\n{class_names[i]} ↔ {class_names[j]}:")
                print(f"  {class_names[i]} → {class_names[j]}: {a_to_b*100:.1f}% (n={cm[i,j]})")
                print(f"  {class_names[j]} → {class_names[i]}: {b_to_a*100:.1f}% (n={cm[j,i]})")
                print(f"  Asymmetry: {asymmetry*100:.1f}pp")
                asymmetry_data.append({
                    'Class_A': class_names[i], 'Class_B': class_names[j],
                    'A_to_B': a_to_b, 'B_to_A': b_to_a, 'Asymmetry': asymmetry,
                    'Abs_A_to_B': cm[i,j], 'Abs_B_to_A': cm[j,i]
                })

    return pd.DataFrame(asymmetry_data)


def calculate_auc_per_class(results):
    """Calculate AUC per class explicitly."""
    print("\n" + "=" * 70)
    print("AUC PER CLASS")
    print("=" * 70)
    n_classes = len(results['class_names'])
    y_true_binary = to_categorical(results['y_true'], num_classes=n_classes)
    auc_values = {}
    for i, class_name in enumerate(results['class_names']):
        fpr, tpr, _ = roc_curve(y_true_binary[:, i], results['y_pred_proba'][:, i])
        roc_auc = auc(fpr, tpr)
        auc_values[class_name] = roc_auc
        print(f"  {class_name}: AUC = {roc_auc:.3f}")
    return auc_values


def optimize_class_thresholds(results, metric='f1'):
    """Optimize decision thresholds per class using validation approach."""
    print("\n" + "=" * 70)
    print(f"THRESHOLD OPTIMIZATION (metric: {metric})")
    print("=" * 70)
    n_classes = len(results['class_names'])
    y_true_binary = to_categorical(results['y_true'], num_classes=n_classes)
    optimized_thresholds = {}
    for i, class_name in enumerate(results['class_names']):
        best_score = 0
        best_thr = 0.5
        for thr in np.linspace(0.1, 0.9, 81):
            y_pred_thr = (results['y_pred_proba'][:, i] >= thr).astype(int)
            if metric == 'f1':
                score = f1_score(y_true_binary[:, i], y_pred_thr)
            elif metric == 'precision':
                score = precision_score(y_true_binary[:, i], y_pred_thr)
            elif metric == 'recall':
                score = recall_score(y_true_binary[:, i], y_pred_thr)
            if score > best_score:
                best_score = score
                best_thr = thr
        optimized_thresholds[class_name] = best_thr
        print(f"  {class_name}: threshold = {best_thr:.3f} (best {metric} = {best_score:.3f})")
    return optimized_thresholds


def apply_thresholds_and_evaluate(results, thresholds):
    """Apply optimized thresholds and re-evaluate."""
    print("\n" + "=" * 70)
    print("EVALUATION WITH OPTIMIZED THRESHOLDS")
    print("=" * 70)
    n_classes = len(results['class_names'])
    y_true_binary = to_categorical(results['y_true'], num_classes=n_classes)
    y_pred_thr = np.zeros_like(results['y_pred_proba'])
    for i, class_name in enumerate(results['class_names']):
        y_pred_thr[:, i] = (results['y_pred_proba'][:, i] >= thresholds[class_name]).astype(int)
    y_pred_new = np.argmax(y_pred_thr, axis=1)
    accuracy = accuracy_score(results['y_true'], y_pred_new)
    precision = precision_score(results['y_true'], y_pred_new, average='weighted')
    recall = recall_score(results['y_true'], y_pred_new, average='weighted')
    f1 = f1_score(results['y_true'], y_pred_new, average='weighted')
    ece = compute_ece(results['y_true'], y_pred_thr)[0]
    print(f"  New Accuracy: {accuracy:.4f}")
    print(f"  New Precision: {precision:.4f}")
    print(f"  New Recall: {recall:.4f}")
    print(f"  New F1-Score: {f1:.4f}")
    print("\nClassification Report (Thresholded):")
    print(classification_report(results['y_true'], y_pred_new, target_names=results['class_names'], digits=4))

    # Return dictionary with all required keys
    return {
        'model_name': f"{results['model_name']}_Thresholded",
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'ece': ece,
        'y_true': results['y_true'],
        'y_pred': y_pred_new,
        'y_pred_proba': y_pred_thr,
        'class_names': results['class_names'],
        'ci_low': results['ci_low'],
        'ci_upp': results['ci_upp'],
        'thresholds': thresholds,
        'uncertainty': results.get('uncertainty'),
        'variance': results.get('variance')
    }

## 13. 🔬 Train MobileNetV2 (Phase1)

In [ ]:
# Create generators for MobileNetV2
mobilenet_train_gen, mobilenet_val_gen, mobilenet_test_gen, mobilenet_class_weights = create_advanced_generators('mobilenet')

In [ ]:
# Build MobileNetV2 model
mobilenet_model, mobilenet_base = build_advanced_model('mobilenet')

In [ ]:
# Train MobileNetV2 with Phase1
mobilenet_history, mobilenet_final_model = train_advanced_model(
    mobilenet_model, mobilenet_base, mobilenet_train_gen, mobilenet_val_gen, mobilenet_test_gen,
    mobilenet_class_weights, model_name='MobileNetV2'
)

# Plot training history
plot_training_history(mobilenet_history.history, 'MobileNetV2')

## 14. Evaluate MobileNetV2

In [ ]:
# Evaluate MobileNetV2 (final) with TTA
mobilenet_results_raw = evaluate_model_advanced(mobilenet_final_model, mobilenet_test_gen, mobilenet_preprocess, 'MobileNetV2', use_tta=True)

# Handle auto-thresholding return
if isinstance(mobilenet_results_raw, tuple):
    mobilenet_results, mobilenet_results_thr = mobilenet_results_raw
else:
    mobilenet_results = mobilenet_results_raw
    mobilenet_results_thr = None
mobilenet_auc = calculate_auc_per_class(mobilenet_results)
plot_confusion_matrix(mobilenet_results, 'MobileNetV2')
plot_roc_curves(mobilenet_results, 'MobileNetV2')
plot_precision_recall_curves(mobilenet_results, 'MobileNetV2')
mobilenet_ece = plot_reliability_diagram(mobilenet_results, 'MobileNetV2')
# Otimizar thresholds para MobileNetV2
mobilenet_thresholds = optimize_class_thresholds(mobilenet_results, metric='f1')
mobilenet_thresholded = apply_thresholds_and_evaluate(mobilenet_results, mobilenet_thresholds)
visualize_misclassifications(mobilenet_results, mobilenet_test_gen, n_examples=9, confidence_threshold=Config.VISUALIZATION_CONFIDENCE_THRESHOLD)

In [ ]:
# Análise de distribuição predita vs real (Comentário 2 do orientador)
print("\n" + "="*70)
print("ANÁLISE DE COLAPSO DE PRIOR - MOBILENETV2")
print("="*70)

# Calcular distribuições
true_counts = np.bincount(mobilenet_results['y_true'])
pred_counts = np.bincount(mobilenet_results['y_pred'])
true_dist = true_counts / len(mobilenet_results['y_true'])
pred_dist = pred_counts / len(mobilenet_results['y_pred'])

print("\n📊 DISTRIBUIÇÃO REAL vs PREDITA:")
print("-" * 70)
for i, class_name in enumerate(Config.CLASS_NAMES):
    shift = pred_dist[i] - true_dist[i]
    shift_pct = (pred_dist[i] / true_dist[i] - 1) * 100
    print(f"{class_name:10s}: Real={true_dist[i]:5.1%} | Predito={pred_dist[i]:5.1%} | "
          f"Shift={shift:+.1%} ({shift_pct:+.1f}% mudança)")

# Identificar colapso de prior
pressure_idx = Config.CLASS_NAMES.index('pressure')
if pred_dist[pressure_idx] / true_dist[pressure_idx] > 1.5:
    print(f"\n⚠️  COLAPSO DE PRIOR DETECTADO: pressure predito {pred_dist[pressure_idx]/true_dist[pressure_idx]:.1f}x acima do real")
else:
    print(f"\n✓ Distribuição de pressure aceitável: {pred_dist[pressure_idx]/true_dist[pressure_idx]:.2f}x do real")

In [ ]:
# Otimizar thresholds especificamente para MobileNetV2
print("=" * 70)
print("MOBILENETV2 - THRESHOLD OPTIMIZATION")
print("=" * 70)
mobilenet_thresholds = optimize_class_thresholds(mobilenet_results, metric='f1')
mobilenet_thresholded = apply_thresholds_and_evaluate(mobilenet_results, mobilenet_thresholds)
plot_confusion_matrix(mobilenet_thresholded, 'MobileNetV2_Thresholded')

# Directional confusion asymmetry analysis for MobileNetV2
print("\n" + "=" * 70)
print("MOBILENETV2 - DIRECTIONAL CONFUSION ASYMMETRY")
print("=" * 70)
mobilenet_asymmetry = analyze_directional_confusion(mobilenet_results, threshold=0.03)
if not mobilenet_asymmetry.empty:
    mobilenet_asymmetry.to_csv(os.path.join(Config.RESULTS_DIR, 'mobilenet_directional_asymmetry.csv'), index=False)
    print("\n✓ Saved per-class asymmetry to mobilenet_directional_asymmetry.csv")

## 16. 🔬 Train EfficientNetB0 (Phase1)

In [ ]:
# Create generators for EfficientNet
effnet_train_gen, effnet_val_gen, effnet_test_gen, effnet_class_weights = create_advanced_generators('efficientnet')

In [ ]:
# Build EfficientNet model
efficientnet_model, efficientnet_base = build_advanced_model('efficientnet')

In [ ]:
f# Train EfficientNet with Phase1
efficientnet_history, efficientnet_final_model = train_advanced_model(
    efficientnet_model, efficientnet_base, effnet_train_gen, effnet_val_gen, effnet_test_gen,
    effnet_class_weights, model_name='EfficientNetB0'
)

# Plot training history with phase separation
plot_training_history(efficientnet_history.history, 'EfficientNetB0')

## 17. Evaluate EfficientNetB0

In [ ]:
# Evaluate EfficientNetV2 (final) with TTA
efficientnet_results_raw = evaluate_model_advanced(efficientnet_final_model, effnet_test_gen, efficientnet_preprocess, 'EfficientNetB0', use_tta=True)

# Handle auto-thresholding return
if isinstance(efficientnet_results_raw, tuple):
    efficientnet_results, efficientnet_results_thr = efficientnet_results_raw
    print(f"\n✓ Threshold optimization applied. Raw and thresholded results available.")
else:
    efficientnet_results = efficientnet_results_raw
    efficientnet_results_thr = None

efficientnet_auc = calculate_auc_per_class(efficientnet_results)
plot_confusion_matrix(efficientnet_results, 'EfficientNetB0')
plot_roc_curves(efficientnet_results, 'EfficientNetB0')
plot_precision_recall_curves(efficientnet_results, 'EfficientNetB0')
effnet_ece = plot_reliability_diagram(efficientnet_results, 'EfficientNetB0')
# Otimizar thresholds para EfficientNetB0
effnet_thresholds = optimize_class_thresholds(efficientnet_results, metric='f1')
effnet_thresholded = apply_thresholds_and_evaluate(efficientnet_results, effnet_thresholds)
visualize_misclassifications(efficientnet_results, effnet_test_gen, n_examples=9, confidence_threshold=0.7)

# Directional confusion asymmetry analysis for EfficientNetB0
print("\n" + "=" * 70)
print("EFFICIENTNETB0 - DIRECTIONAL CONFUSION ASYMMETRY")
print("=" * 70)
efficientnet_asymmetry = analyze_directional_confusion(efficientnet_results)
if not efficientnet_asymmetry.empty:
    efficientnet_asymmetry.to_csv(os.path.join(Config.RESULTS_DIR, 'efficientnet_directional_asymmetry.csv'), index=False)
    print("\n✓ Saved per-class asymmetry to efficientnet_directional_asymmetry.csv")

In [ ]:
# Análise de distribuição predita vs real - EfficientNetB0
print("\n" + "="*70)
print("ANÁLISE DE COLAPSO DE PRIOR - EFFICIENTNETB0")
print("="*70)

# Calcular distribuições
true_counts = np.bincount(efficientnet_results['y_true'])
pred_counts = np.bincount(efficientnet_results['y_pred'])
true_dist = true_counts / len(efficientnet_results['y_true'])
pred_dist = pred_counts / len(efficientnet_results['y_pred'])

print("\n📊 DISTRIBUIÇÃO REAL vs PREDITA:")
print("-" * 70)
for i, class_name in enumerate(Config.CLASS_NAMES):
    shift = pred_dist[i] - true_dist[i]
    shift_pct = (pred_dist[i] / true_dist[i] - 1) * 100
    print(f"{class_name:10s}: Real={true_dist[i]:5.1%} | Predito={pred_dist[i]:5.1%} | "
          f"Shift={shift:+.1%} ({shift_pct:+.1f}% mudança)")

# Identificar colapso de prior
pressure_idx = Config.CLASS_NAMES.index('pressure')
if pred_dist[pressure_idx] / true_dist[pressure_idx] > 1.5:
    print(f"\n⚠️  COLAPSO DE PRIOR DETECTADO: pressure predito {pred_dist[pressure_idx]/true_dist[pressure_idx]:.1f}x acima do real")
else:
    print(f"\n✓ Distribuição de pressure aceitável: {pred_dist[pressure_idx]/true_dist[pressure_idx]:.2f}x do real")

#  Comparação direta MobileNetV2 vs EfficientNetB0
print("\n" + "="*70)
print("COMPARAÇÃO: MELHORIA DE FP(pressure)")
print("="*70)
analyze_class_distribution_shift([mobilenet_results, efficientnet_results])

In [ ]:
# ===========================================
# COMPARAÇÃO DE FALSOS POSITIVOS - PRESSURE
# ===========================================

print("\n" + "="*70)
print("FALSE POSITIVE COMPARISON - PRESSURE CLASS")
print("="*70)

def compare_fp_pressure(results_list):
    fp_data = []
    for r in results_list:
        cm = confusion_matrix(r['y_true'], r['y_pred'])
        pressure_idx = r['class_names'].index('pressure')
        fp = np.sum(cm[:, pressure_idx]) - cm[pressure_idx, pressure_idx]
        fp_data.append({'Model': r['model_name'], 'FP_Pressure': fp})

    df = pd.DataFrame(fp_data)
    print(df.to_string(index=False))

    if len(df) == 2:
        reduction = df.loc[0, 'FP_Pressure'] - df.loc[1, 'FP_Pressure']
        pct_reduction = reduction / df.loc[0, 'FP_Pressure'] * 100
        print(f"\n✓ Redução de FP: {reduction} amostras ({pct_reduction:.1f}%)")
    return df

fp_comparison = compare_fp_pressure([mobilenet_results, efficientnet_results])
fp_comparison.to_csv(os.path.join(Config.RESULTS_DIR, 'false_positive_comparison.csv'), index=False)

In [ ]:
# ===========================================
# AJUSTE 3: EXECUTAR ANÁLISE DE ENSEMBLE
# ===========================================

print("\n" + "=" * 70)
print("AJUSTE 3: EXECUTANDO ANÁLISE DE ENSEMBLE")
print("=" * 70)

# Create ensemble with calibration-based weighting
ensemble_results = ensemble_predictions([mobilenet_results, efficientnet_results],
                                       weights=[0.4, 0.6])  # Pesos baseados em performance

# Análise de trade-offs por classe
ensemble_cm = confusion_matrix(ensemble_results['y_true'], ensemble_results['y_pred'])
mobile_cm = confusion_matrix(mobilenet_results['y_true'], mobilenet_results['y_pred'])
effnet_cm = confusion_matrix(efficientnet_results['y_true'], efficientnet_results['y_pred'])

print("\nEnsemble Trade-off Analysis per Class:")
print("-" * 70)
for i, class_name in enumerate(Config.CLASS_NAMES):
    # Recall changes
    mobile_recall = mobile_cm[i,i] / mobile_cm[i,:].sum()
    effnet_recall = effnet_cm[i,i] / effnet_cm[i,:].sum()
    ensemble_recall = ensemble_cm[i,i] / ensemble_cm[i,:].sum()

    print(f"{class_name}: recall {mobile_recall:.3f}→{ensemble_recall:.3f} ({ensemble_recall-mobile_recall:+.3f})")

# Visualizações do ensemble
plot_confusion_matrix(ensemble_results, 'Ensemble')
plot_roc_curves(ensemble_results, 'Ensemble')
plot_precision_recall_curves(ensemble_results, 'Ensemble')
plot_reliability_diagram(ensemble_results, 'Ensemble')

In [ ]:
# ===========================================
# 🔬 CÉLULA ÚNICA DE DIAGNÓSTICO COMPLETO
# ===========================================

print("\n" + "="*70)
print("🔬 EXECUTANDO DIAGNÓSTICO COMPLETO")
print("="*70)

# Diagnóstico de sink class
print("\n[1/4] Análise de Sink Class...")
mobilenet_sink = diagnose_pressure_sink(mobilenet_results)
effnet_sink = diagnose_pressure_sink(efficientnet_results)

# Comparação de FP
print("\n[2/4] Comparação de Falsos Positivos...")
fp_comparison = compare_fp_pressure([mobilenet_results, efficientnet_results])

# Análise direcional detalhada
print("\n[3/4] Assimetria Direcional...")
mobilenet_asym = analyze_directional_confusion(mobilenet_results, threshold=0.03)
effnet_asym = analyze_directional_confusion(efficientnet_results, threshold=0.03)

# Salvamento consolidado
print("\n[4/4] Salvando diagnósticos...")
mobilenet_sink.to_csv(os.path.join(Config.RESULTS_DIR, 'mobilenet_sink_analysis.csv'))
effnet_sink.to_csv(os.path.join(Config.RESULTS_DIR, 'efficientnet_sink_analysis.csv'))
fp_comparison.to_csv(os.path.join(Config.RESULTS_DIR, 'false_positive_comparison.csv'))
print(f"✓ Diagnósticos salvos em {Config.RESULTS_DIR}")

# =============================================================

## 19. Ensemble Analysis with Per-Class Impact

In [ ]:
# ===========================================
# EXECUTAR ENSEMBLE APÓS OTIMIZAÇÃO DE THRESHOLDS
# ===========================================

# Trecho deve ser inserido APÓS as células de threshold (seções 15 e 17)
# e ANTES da análise Macro vs Weighted F1 (seção 20)

# Create ensemble with calibration-based weighting

print("\n" + "=" * 70)
print("ENSEMBLE ANALYSIS - EXECUTING AFTER THRESHOLD OPTIMIZATION")
print("=" * 70)

# Weight based on accuracy * (1 - ECE) for better calibration
weights = [
    mobilenet_results['accuracy'] * (1 - mobilenet_results['ece']),
    efficientnet_results['accuracy'] * (1 - efficientnet_results['ece'])
]
weights = np.array(weights) / np.sum(weights)

ensemble_results = ensemble_predictions([mobilenet_results, efficientnet_results], weights=weights)

# Compare per-class changes
print("\n" + "=" * 70)
print("ENSEMBLE vs INDIVIDUAL PER-CLASS PERFORMANCE")
print("=" * 70)
cm_mobilenet = confusion_matrix(mobilenet_results['y_true'], mobilenet_results['y_pred'])
cm_effnet = confusion_matrix(efficientnet_results['y_true'], efficientnet_results['y_pred'])
cm_ensemble = confusion_matrix(ensemble_results['y_true'], ensemble_results['y_pred'])

per_class_changes = []
for i, class_name in enumerate(Config.CLASS_NAMES):
    # Recall
    recall_mobilenet = cm_mobilenet[i, i] / np.sum(cm_mobilenet[i, :])
    recall_effnet = cm_effnet[i, i] / np.sum(cm_effnet[i, :])
    recall_ensemble = cm_ensemble[i, i] / np.sum(cm_ensemble[i, :])

    # Precision
    precision_mobilenet = cm_mobilenet[i, i] / np.sum(cm_mobilenet[:, i]) if np.sum(cm_mobilenet[:, i]) > 0 else 0
    precision_effnet = cm_effnet[i, i] / np.sum(cm_effnet[:, i]) if np.sum(cm_effnet[:, i]) > 0 else 0
    precision_ensemble = cm_ensemble[i, i] / np.sum(cm_ensemble[:, i]) if np.sum(cm_ensemble[:, i]) > 0 else 0

    # F1
    f1_mobilenet = 2 * (precision_mobilenet * recall_mobilenet) / (precision_mobilenet + recall_mobilenet) if (precision_mobilenet + recall_mobilenet) > 0 else 0
    f1_effnet = 2 * (precision_effnet * recall_effnet) / (precision_effnet + recall_effnet) if (precision_effnet + recall_effnet) > 0 else 0
    f1_ensemble = 2 * (precision_ensemble * recall_ensemble) / (precision_ensemble + recall_ensemble) if (precision_ensemble + recall_ensemble) > 0 else 0

    # Changes from EfficientNet to Ensemble
    recall_change = recall_ensemble - recall_effnet
    precision_change = precision_ensemble - precision_effnet
    f1_change = f1_ensemble - f1_effnet

    per_class_changes.append({
        'Class': class_name,
        'Recall_EfficientNet': recall_effnet,
        'Recall_Ensemble': recall_ensemble,
        'Recall_Change': recall_change,
        'Precision_EfficientNet': precision_effnet,
        'Precision_Ensemble': precision_ensemble,
        'Precision_Change': precision_change,
        'F1_EfficientNet': f1_effnet,
        'F1_Ensemble': f1_ensemble,
        'F1_Change': f1_change
    })

    print(f"\n{class_name}:")
    print(f"  Recall: {recall_effnet:.3f} → {recall_ensemble:.3f} ({recall_change:+.3f})")
    print(f"  Precision: {precision_effnet:.3f} → {precision_ensemble:.3f} ({precision_change:+.3f})")
    print(f"  F1-Score: {f1_effnet:.3f} → {f1_ensemble:.3f} ({f1_change:+.3f})")

# Visualizar ensemble confusion matrix
plot_confusion_matrix(ensemble_results, 'Ensemble')
plot_roc_curves(ensemble_results, 'Ensemble')
plot_precision_recall_curves(ensemble_results, 'Ensemble')
plot_reliability_diagram(ensemble_results, 'Ensemble')

per_class_df = pd.DataFrame(per_class_changes)
per_class_df.to_csv(os.path.join(Config.RESULTS_DIR, 'per_class_ensemble_impact.csv'), index=False)
print("\n✓ Saved per-class changes to per_class_ensemble_impact.csv")

# ===========================================
# FIM DO ENSEMBLE
# ===========================================

In [ ]:
# ===========================================
# AJUSTE 6: SALVAR RESULTADOS DO ENSEMBLE
# ===========================================

# Persistir análise de trade-offs por classe
per_class_df = pd.DataFrame(per_class_changes)
per_class_df.to_csv(os.path.join(Config.RESULTS_DIR, 'ensemble_per_class_impact.csv'), index=False)

# Salvar métricas completas do ensemble
ensemble_summary = {
    'Model': 'Ensemble',
    'Accuracy': ensemble_results['accuracy'],
    'Macro_F1': f1_score(ensemble_results['y_true'], ensemble_results['y_pred'], average='macro'),
    'Weighted_F1': ensemble_results['f1_score'],
    'ECE': ensemble_results['ece'],
    'Pressure_Recall': ensemble_cm[1,1] / ensemble_cm[1,:].sum(),
    'Pressure_Precision': ensemble_cm[1,1] / ensemble_cm[:,1].sum(),
}
pd.DataFrame([ensemble_summary]).to_csv(os.path.join(Config.RESULTS_DIR, 'ensemble_summary.csv'), index=False)

print(f"\n✓ Salvos ensemble_per_class_impact.csv e ensemble_summary.csv")
print(f"✓ Diretório: {Config.RESULTS_DIR}")

In [ ]:
print("\n" + "="*70)
print("📊 TRADE-OFFS QUANTIFICADOS DO ENSEMBLE")
print("="*70)

for change in per_class_changes:
    print(f"\n{change['Class']}:")
    print(f"  Recall: {change['Recall_EfficientNet']:.3f} → {change['Recall_Ensemble']:.3f} "
          f"({change['Recall_Change']:+.3f})")
    print(f"  Precision: {change['Precision_EfficientNet']:.3f} → {change['Precision_Ensemble']:.3f} "
          f"({change['Precision_Change']:+.3f})")

print("\n✅ Trade-offs salvos em per_class_ensemble_impact.csv")

## 20. Macro vs Weighted F1 Analysis

In [ ]:
# Calculate Macro vs Weighted F1
print("=" * 70)
print("MACRO vs WEIGHTED F1 ANALYSIS")
print("=" * 70)

def calculate_macro_weighted_f1(results):
    y_true = results['y_true']
    y_pred = results['y_pred']
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')
    gap = weighted_f1 - macro_f1
    return macro_f1, weighted_f1, gap

f1_analysis = []
for results in [mobilenet_results, efficientnet_results, ensemble_results]:
    macro, weighted, gap = calculate_macro_weighted_f1(results)
    f1_analysis.append({'Model': results['model_name'], 'Macro_F1': macro, 'Weighted_F1': weighted, 'Gap': gap})
    print(f"{results['model_name']}: Macro={macro:.4f}, Weighted={weighted:.4f}, Gap={gap:+.4f}")

f1_df = pd.DataFrame(f1_analysis)
print("\n✓ Gap interpretation: Positive gap indicates better performance on majority classes")
print("✗ Large gap suggests fragility on minority classes (pressure, surgical)")

# Visualizar gap
fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(f1_df))
width = 0.35
ax.bar(x - width/2, f1_df['Macro_F1'], width, label='Macro F1', color='skyblue')
ax.bar(x + width/2, f1_df['Weighted_F1'], width, label='Weighted F1', color='darkblue')
ax.set_xlabel('Model')
ax.set_ylabel('F1-Score')
ax.set_title('Macro vs Weighted F1 Comparison')
ax.set_xticks(x)
ax.set_xticklabels(f1_df['Model'], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'macro_weighted_f1_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Validação automática do Gap Macro-Weighted

print("\n" + "="*70)
print("VALIDAÇÃO DO GAP MACRO-WEIGHTED")
print("="*70)

THRESHOLD_GAP = 0.02

for results in [mobilenet_results, efficientnet_results, ensemble_results]:
    macro_f1, weighted_f1, gap = calculate_macro_weighted_f1(results)
    status = "✅ ACEITÁVEL" if gap < THRESHOLD_GAP else "❌ VIÉS SEVERO"
    print(f"{results['model_name']}: gap={gap:.4f} {status}")

print(f"\nThreshold: gap < {THRESHOLD_GAP} para ausência de viés severo")

## 21. Statistical Comparison (McNemar's Test)

In [ ]:
# Compare MobileNetV2 vs EfficientNetB0
mcnemar_result, contingency_table = compare_models_mcnemar(mobilenet_results, efficientnet_results)

## 22. Final Model Comparison with All Metrics

In [ ]:
# Comprehensive final comparison
print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON - PHASE 1 ONLY")
print("=" * 70)

final_comparison = []
for r in [mobilenet_results, efficientnet_results, ensemble_results]:
    macro_f1, weighted_f1, gap = calculate_macro_weighted_f1(r)

    # Calculate per-class F1
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    class_f1 = []
    for i, class_name in enumerate(Config.CLASS_NAMES):
        precision = cm[i, i] / np.sum(cm[:, i]) if np.sum(cm[:, i]) > 0 else 0
        recall = cm[i, i] / np.sum(cm[i, :]) if np.sum(cm[i, :]) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        class_f1.append(f1)

    final_comparison.append({
        'Model': r['model_name'],
        'Accuracy': f"{r['accuracy']:.1%}",
        'Precision': f"{r['precision']:.3f}",
        'Recall': f"{r['recall']:.3f}",
        'Weighted_F1': f"{weighted_f1:.3f}",
        'Macro_F1': f"{macro_f1:.3f}",
        'Gap': f"{gap:+.4f}",
        'ECE': f"{r['ece']:.4f}",
        'Diabetic_F1': f"{class_f1[0]:.3f}",
        'Pressure_F1': f"{class_f1[1]:.3f}",
        'Surgical_F1': f"{class_f1[2]:.3f}",
        'Venous_F1': f"{class_f1[3]:.3f}"
    })

final_df = pd.DataFrame(final_comparison)
print(final_df.to_string(index=False))

# Save final comparison
final_df.to_csv(os.path.join(Config.RESULTS_DIR, 'final_model_comparison_v6.csv'), index=False)
print(f"\n✓ Saved final comparison to: {os.path.join(Config.RESULTS_DIR, 'final_model_comparison_v6.csv')}")

In [ ]:
# Aplicar temperature scaling explícito nos modelos finais
from sklearn.calibration import CalibratedClassifierCV

def apply_temperature_scaling(results, model, test_gen, preprocess_func):
    """Apply temperature scaling to calibrated probabilities"""
    print(f"\nApplying Temperature Scaling to {results['model_name']}...")

    # Obter logits (pre-softmax) - necessário modificar evaluate_model_advanced para retornar logits
    # OU usar abordagem simplificada: ajustar temperatura nas probabilidades

    # Abordagem simplificada (ajuste post-hoc):
    from scipy.optimize import minimize

    def ece_loss(T):
        scaled_probs = np.exp(np.log(results['y_pred_proba'] + 1e-10) / T[0])
        scaled_probs = scaled_probs / np.sum(scaled_probs, axis=1, keepdims=True)
        ece, _ = compute_ece(results['y_true'], scaled_probs)
        return ece

    # Otimizar temperatura
    result = minimize(ece_loss, [1.0], bounds=[(0.1, 10.0)])
    best_temp = result.x[0]

    # Aplicar temperatura otimizada
    calibrated_probs = np.exp(np.log(results['y_pred_proba'] + 1e-10) / best_temp)
    calibrated_probs = calibrated_probs / np.sum(calibrated_probs, axis=1, keepdims=True)

    # Re-avaliar
    y_pred_calibrated = np.argmax(calibrated_probs, axis=1)
    accuracy = accuracy_score(results['y_true'], y_pred_calibrated)
    ece, _ = compute_ece(results['y_true'], calibrated_probs)

    print(f"Optimal temperature: {best_temp:.3f}")
    print(f"Calibrated accuracy: {accuracy:.4f}")
    print(f"Calibrated ECE: {ece:.4f}")

    return calibrated_probs, best_temp

# Aplicar para ambos os modelos
mobilenet_calibrated, mobilenet_temp = apply_temperature_scaling(mobilenet_results, mobilenet_final_model, mobilenet_test_gen, mobilenet_preprocess)
effnet_calibrated, effnet_temp = apply_temperature_scaling(efficientnet_results, efficientnet_final_model, effnet_test_gen, efficientnet_preprocess)

## 23. Efficiency Trade-off Analysis

In [ ]:
# Plotar accuracy vs inference time dos modelos TREINADOS
print("\n" + "=" * 70)
print("EFFICIENCY TRADE-OFF (SIMPLIFIED)")
print("=" * 70)

models = ['MobileNetV2', 'EfficientNetB0']
accuracies = [mobilenet_results['accuracy'], efficientnet_results['accuracy']]

# Medir tempo de inferência (simplificado)
import time

inference_times = []
for model, name in zip([mobilenet_final_model, efficientnet_final_model], models):
    dummy_input = np.random.randn(1, Config.IMG_SIZE, Config.IMG_SIZE, 3).astype(np.float32)

    # Warmup
    for _ in range(10):
        _ = model.predict(dummy_input, verbose=0)

    # Measure
    times = []
    for _ in range(50):
        start = time.time()
        _ = model.predict(dummy_input, verbose=0)
        times.append(time.time() - start)

    avg_time = np.mean(times) * 1000  # ms
    inference_times.append(avg_time)
    print(f"{name}: {avg_time:.2f}ms inference time")

# Plotar
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(inference_times, accuracies, s=200, alpha=0.6, c=['blue', 'green'])

for i, model in enumerate(models):
    ax.annotate(model, (inference_times[i], accuracies[i]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=12, fontweight='bold')

ax.set_xlabel('Inference Time (ms)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy vs Inference Time Trade-off')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(Config.RESULTS_DIR, 'accuracy_vs_inference.png'), dpi=150)
plt.show()

## 25. CHANGELOG

In [ ]:
changelog = """
CHANGELOG
=====================================================

[NEW] Config.UNFREEZE_LR_FACTOR = 0.1
      - Technical: Reduces LR by 10x when unfreezing backbone
      - Justification: Prevents feature destruction during fine-tuning

[NEW] train_advanced_model() now returns phase1_results and phase2_results
      - Technical: Saves and evaluates both checkpoints
      - Justification: Enables direct comparison Phase1-only vs Phase1+Phase2

[NEW] analyze_class_distribution_shift()
      - Technical: Compares true vs predicted class proportions
      - Justification: Quantifies posterior shift and sink class behavior

[NEW] analyze_false_positives_sink_class()
      - Technical: Counts FP by class, especially for sink class
      - Justification: Identifies systematic biases in predictions

[NEW] calculate_auc_per_class()
      - Technical: Explicit AUC calculation per class
      - Justification: Separates ranking capacity from thresholding issues

[NEW] optimize_class_thresholds() and apply_thresholds_and_evaluate()
      - Technical: F1-optimized thresholds per class
      - Justification: Demonstrates recoverability without changing backbone

[NEW] analyze_directional_confusion()
      - Technical: Quantifies confusion asymmetry between class pairs
      - Justification: Identifies systematic vs random errors

[MODIFIED] Ensemble weights now use accuracy * (1 - ECE)
      - Technical: Prefers well-calibrated models
      - Justification: Reduces overconfident but wrong predictions

[NEW] calculate_macro_weighted_f1() analysis
      - Technical: Reports both macro and weighted F1 with gap
      - Justification: Highlights equity issues across classes

[NEW] Comprehensive final comparison table
      - Technical: Includes per-class F1, AUC, and calibration
      - Justification: Provides single source of truth for all metrics

[MODIFIED] PHASE 2 REMOVIDO DO PIPELINE
      - Motivo: Instabilidade observada (queda de accuracy em época ~40)
      - Evidência: Figuras 1 e 2 mostram deterioração em Phase 2
      - Decisão: Treinar apenas Phase 1 (feature extraction)
      - Impacto: Melhor estabilidade, sem perda de performance final

FILES GENERATED:
- final_model_comparison_v6.csv
- macro_weighted_f1_comparison.png
- MobileNetV2_*.png (Phase1 vs Phase2)
- EfficientNetB0_*.png (Phase1 vs Phase2)
- *reliability.png (ECE diagnostics)
- *confusion_matrix.png (8 matrices total)
- *roc_curves.png (AUC visualization)
- *pr_curves.png (precision-recall)
- accuracy_vs_inference.png (efficiency trade-off)
"""

print(changelog)

## 26. Download Results

In [ ]:
# Zip and download all results
!zip -r /content/wound_classification_results_v6_corrections.zip \
    {Config.RESULTS_DIR} {Config.MODEL_SAVE_DIR} {Config.DATA_DIR}/dataset_metadata.csv \
    -x "*/.ipynb_checkpoints/*" -x "*/__pycache__/*"

from google.colab import files
files.download('/content/wound_classification_results_v6_corrections.zip')

print("\n" + "=" * 70)
print("PIPELINE EXECUTION COMPLETE")
print("=" * 70)
print(f"Results saved to: {Config.RESULTS_DIR}")
print(f"Models saved to: {Config.MODEL_SAVE_DIR}")
print(f"Zip file: /content/wound_classification_results_v6_corrections.zip")
print("=" * 70)